# TrendLens — 07 · RAG + API + Frontend Wiring (Phase 7)

Stage 13: a query is embedded with CLIP, top-k clusters retrieved from the Phase 6 FAISS index, and an **honest** markdown answer is assembled from real pipeline artifacts (interpretations, trend metrics, representatives). No LLM, no fabricated numbers.

> **Integrity:** timestamps/engagement are neutral synthetic (demo); cluster names are VLM interpretations, not ground truth; anything not measured is shown as `—`/null, never invented.

In [1]:
import sys
from pathlib import Path
REPO = Path.cwd()
if not (REPO / "config.py").exists():
    for p in Path.cwd().parents:
        if (p / "config.py").exists():
            REPO = p; break
sys.path.insert(0, str(REPO))

from src import rag

## 1 · A query, end to end

In [2]:
res = rag.run_query("dogs on the sofa", k=3)
print(res["answer"])

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 📊 Trend summary
Retrieved **3 clusters** from the FAISS index. Top themes: **book, chair, dog, laying, reading, sitting**.
Lifecycle: 📈 3 Rising

## 📈 Leading cluster — #0 (Rising)
**Name (VLM interpretation):** dog laying
**Description:** A visual cluster whose images are described by: dog, laying, blanket, couch, green, small
**Visual evidence:** “a dog laying on a green blanket”
**Metrics:** 168 posts · avg engagement 71.17 · recent growth 0.60 · trend score 0.06
**Interpretation confidence:** 0.08

## 📡 All retrieved clusters
| Rank | Cluster | Name | Lifecycle | Posts | Engagement | Trend score | Conf. |
|------|---------|------|-----------|-------|------------|-------------|-------|
| #1 | 0 | dog laying | 📈 Rising | 168 | 71.17 | 0.06 | 0.08 |
| #2 | 16 | sitting chair | 📈 Rising | 212 | 75.40 | 0.12 | 0.04 |
| #3 | 17 | reading book | 📈 Rising | 115 | 83.22 | 0.02 | 0.05 |

*Data source: TrendLens pipeline — CLIP clustering of 5,000 sampled images, BLIP interpretations (not 

In [3]:
import pandas as pd
rows = [{"cluster": c["cluster_id"], "rank": c["rank"], "name": c["name"],
         "lifecycle": c["lifecycle"], "posts": c["n_posts"],
         "engagement": round(c["average_engagement"], 2) if c["average_engagement"] is not None else None,
         "conf": c["interpretation_confidence"],
         "similarity": c["similarity_score"]} for c in res["retrievedClusters"]]
pd.DataFrame(rows)

,cluster,rank,name,lifecycle,posts,engagement,conf,similarity
0,0,1,dog laying,Rising,168,71.17,0.0792,0.7991
1,16,2,sitting chair,Rising,212,75.40,0.0389,0.7106
2,17,3,reading book,Rising,115,83.22,0.0451,0.7012


## 2 · The API server (stdlib, no new deps)

Start it with `python -m src.api` — exposes `/api/health`, `/api/rag-query`, `/api/trends`, `/api/clusters`, `/api/predict-popularity` (honest reference stats only).

In [4]:
from src import api
print("health:", api.handle_health()["status"], "| clusters:", api.handle_health()["totalClustersAnalyzed"])
print("predict:", api.handle_predict({"clusterId": 0})["status"])

health: ok | clusters: 29
predict: NOT EVALUATED


## 3 · Frontend wiring

`frontend/server.ts` now **proxies every `/api/*` call to the Python backend** and serves the React app via Vite. It contains zero fabricated demo clusters — if the backend is offline it returns an explicit `backend-offline` message instead of inventing data.

Run everything:
```bash
./scripts/run_all.sh
#   Python backend: http://127.0.0.1:8000/api/health
#   React frontend: http://127.0.0.1:3000
```

In [5]:
print("done — Phase 7 ready for human review in the browser.")

done — Phase 7 ready for human review in the browser.


## Phase 7 checkpoint
- [x] `src/rag.py` — CLIP+FAISS retrieval → honest markdown answer (no LLM)
- [x] `src/api.py` — stdlib JSON API (health/rag-query/trends/clusters/predict)
- [x] React `server.ts` — proxy to Python backend; fabricated demo clusters removed
- [x] Run scripts + honest offline message

**Next (Phase 8):** end-to-end evaluation — lead-time experiment (visual vs text trend signal) using the Phase 4 text baseline, plus a README/summary.